# Retrain P13 with more seeds

Runs the P13 recipe (`temporal_model/train.py`) with five new seeds on every labeled trace, for an ensemble with the original P13.
Runtime: GPU (Runtime -> Change runtime type -> T4 GPU). About 15 minutes in total.

Run this notebook in the Colab web UI ([open in Colab](https://colab.research.google.com/github/Agata872/Huawei/blob/main/colab/train_p13.ipynb)): it needs one file uploaded and returns a zip, and the upload and download helpers only work there.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


Clone the code (public repository, `main` branch), or update an existing clone.

In [2]:
import os, subprocess
if not os.path.exists('/content/Huawei'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Agata872/Huawei.git', '/content/Huawei'], check=True)
%cd /content/Huawei
!git pull -q
!git log --oneline -1
!pip install -q 'blosc2>=3,<5'

/content/Huawei
8018e30 (grafted, HEAD -> main, origin/main, origin/HEAD) Add analysis scripts for room types and seed agreement


Download the labeled archive (2.1 GB) from the link in `FutureNetwork-starting-kit/data_downloads.md`, check its SHA-256 and extract it.

In [3]:
%%bash
set -e
mkdir -p data/downloads data/extracted
cd data/downloads
[ -f FutureNetwork-data-labeled.zip ] || wget -q https://dl.r0h.eu/techarena2026-paris/FutureNetwork-data-labeled.zip
echo '1f60df2d4c1459c307e444741056e17a5ec537614cf577331ed04eb94d0765c1  FutureNetwork-data-labeled.zip' | sha256sum -c
unzip -q -n FutureNetwork-data-labeled.zip -d ../extracted
ls ../extracted/labeled | wc -l

FutureNetwork-data-labeled.zip: OK
97


Precompute the block maps (CPU, a few minutes) and check them against the raw-input features: the printed difference should be ~1e-6 or smaller.

In [4]:
!python -m temporal_model.maps --check

[1/48] a_a1_hash813694_1
[2/48] a_a1_hash058655_2
[3/48] a_a1_hash240174_2
[4/48] a_a1_hash861722_3
[5/48] a_a1_hash033659_3
[6/48] a_a1_hash844151_4
[7/48] a_a1_hash330776_4
[8/48] a_a1_hash420651_4
[9/48] a_a1_hash280746_3
[10/48] a_a1_hash069403_2
[11/48] a_a1_hash221231_1
[12/48] a_a1_hash957492_0
[13/48] c_c1_hash906339_1
[14/48] c_c1_hash072792_2
[15/48] c_c1_hash810891_3
[16/48] c_c1_hash292683_4
[17/48] c_c1_hash368203_4
[18/48] c_c1_hash672642_4
[19/48] c_c1_hash534150_4
[20/48] c_c1_hash419093_3
[21/48] c_c1_hash712526_3
[22/48] c_c1_hash884642_2
[23/48] c_c1_hash562262_2
[24/48] c_c1_hash347235_1
[25/48] b_b1_hash962037_0
[26/48] e_e1_hash508136_0
[27/48] e_e1_hash657193_0
[28/48] e_e1_hash250867_1
[29/48] unknown_unknown_hash287936_1
[30/48] unknown_unknown_hash461239_1
[31/48] unknown_unknown_hash081247_1
[32/48] g_g1_hash748206_0
[33/48] g_g1_hash299607_1
[34/48] g_g1_hash245884_1
[35/48] g_g1_hash284913_1
[36/48] g_g1_hash352161_0
[37/48] g_g1_hash335239_1
[38/48] g_g1_h

Upload the P11B checkpoint from your local copy of the Reference folder:
`Reference/roomsense_p13_fast/experiments/runs_p11_fast/P11B/checkpoint.pt` (3 MB).
It is not in the repository and only lives in this runtime. Instead of the file picker you can also drag it into `/content/` with the file browser and rename it to `P11B_checkpoint.pt`.

In [5]:
import os, shutil
P11B = '/content/P11B_checkpoint.pt'
if not os.path.exists(P11B):
    from google.colab import files
    name, = files.upload()
    shutil.move(name, P11B)
!echo 'c6ae4c5814cea75f3522d99ed7e014c7ae2612eb2e49a8cb4a8d7dbb9d42fb82  /content/P11B_checkpoint.pt' | sha256sum -c

KeyboardInterrupt: 

Train seeds 1-5 (about a minute each). Each run prints its loss every 400 steps.

In [ ]:
!python -m temporal_model.train --p11b /content/P11B_checkpoint.pt --seeds 1,2,3,4,5

Download the five checkpoints and unzip them into the local repository's `runs/temporal_model/`.
Do this before disconnecting: the runtime's files are deleted with it. A copy also goes to Google Drive (`MyDrive/huawei/p13_seeds.zip`) if you allow the Drive connection.

In [ ]:
!rm -f /content/p13_seeds.zip && zip -qj /content/p13_seeds.zip runs/temporal_model/p13_seed*.pt && ls -l /content/p13_seeds.zip
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/huawei', exist_ok=True)
    shutil.copy('/content/p13_seeds.zip', '/content/drive/MyDrive/huawei/p13_seeds.zip')
    print('copied to Google Drive: MyDrive/huawei/p13_seeds.zip')
except Exception as error:
    print(f'Google Drive unavailable ({error})')
try:
    from google.colab import files
    files.download('/content/p13_seeds.zip')
except Exception as error:
    print(f'browser download unavailable ({error}); fetch /content/p13_seeds.zip from the file browser')